In [2]:
import pandas as pd
import numpy as np
from transformers import pipeline

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("../Outputs/merged_text_905.csv")
print(f"Loaded: {df.shape[0]} rows")
print(f"Posts with usable text: {df['has_text'].sum()}")

Loaded: 905 rows
Posts with usable text: 797


In [4]:
# Twitter-trained model. 11 emotions, multi-label.
emotion_model = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-emotion-multilabel-latest",
    top_k=None,
    truncation=True
)

# Quick test
test = emotion_model("I'm so excited for the Super Bowl!")
print(f"Number of labels: {len(test[0])}")
print(f"All labels: {[item['label'] for item in test[0]]}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15166.95it/s]


Number of labels: 11
All labels: ['joy', 'optimism', 'anticipation', 'love', 'trust', 'surprise', 'fear', 'anger', 'disgust', 'pessimism', 'sadness']


In [5]:
texts = df.loc[df["has_text"], "text_for_analysis"].tolist()
texts = [t[:2000] for t in texts]

print(f"Running Cardiff Twitter emotion model on {len(texts)} posts...")
results = emotion_model(texts, batch_size=16)
print(f"Got {len(results)} results")

Running Cardiff Twitter emotion model on 797 posts...
Got 797 results


In [6]:
emotion_labels = sorted([item["label"] for item in results[0]])
emo_cols = [f"emo_{c}" for c in emotion_labels]
print(f"Labels: {emotion_labels}")
print(f"Total: {len(emotion_labels)} emotions")

Labels: ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']
Total: 11 emotions


In [7]:
# Drop existing emotion columns if they exist (safe re-run)
df = df.drop(columns=[c for c in emo_cols + ["dominant_emotion", "dominant_emotion_score"] if c in df.columns])

# Build score dataframe
rows = [{item["label"]: item["score"] for item in res} for res in results]
emo_df = pd.DataFrame(rows)[emotion_labels]
emo_df.columns = emo_cols
emo_df.index = df.index[df["has_text"]]
df = df.join(emo_df)

# Compute dominant emotion (only on rows with text)
df["dominant_emotion"] = None
df["dominant_emotion_score"] = None
df.loc[df["has_text"], "dominant_emotion"] = (
    df.loc[df["has_text"], emo_cols].idxmax(axis=1).str.replace("emo_", "")
)
df.loc[df["has_text"], "dominant_emotion_score"] = df.loc[df["has_text"], emo_cols].max(axis=1)

print(df[["student_id", "text_source", "dominant_emotion", "dominant_emotion_score"]].head())

  student_id         text_source dominant_emotion dominant_emotion_score
0        1_A  caption+transcript          disgust               0.819981
1        1_A  caption+transcript         optimism               0.977945
2        1_A        caption_only              joy               0.746067
3        2_A        caption_only     anticipation               0.692761
4        2_A        caption_only     anticipation               0.857154


In [8]:
print("=== Overall dominant emotion counts ===")
print(df["dominant_emotion"].value_counts(dropna=False))
print()
print("=== Mean scores across all posts ===")
print(df[emo_cols].mean().sort_values(ascending=False))

=== Overall dominant emotion counts ===
dominant_emotion
joy             362
anticipation    188
None            108
optimism         80
sadness          69
anger            44
disgust          29
fear             25
Name: count, dtype: int64

=== Mean scores across all posts ===
emo_joy             0.572869
emo_optimism        0.405202
emo_anticipation    0.365263
emo_sadness         0.142601
emo_love            0.138896
emo_disgust         0.132117
emo_anger           0.111819
emo_trust           0.099919
emo_surprise        0.092611
emo_fear            0.086698
emo_pessimism       0.054297
dtype: float64


In [9]:
for emo in emotion_labels:
    subset = df[df["dominant_emotion"] == emo]
    print(f"\n=== {emo.upper()} ({len(subset)} posts) ===")
    if len(subset) == 0:
        continue
    top = subset.nlargest(3, f"emo_{emo}")
    for _, r in top.iterrows():
        text_preview = r['text_for_analysis'][:180].replace('\n', ' | ')
        print(f"  [{r[f'emo_{emo}']:.2f}] [{r['text_source']}] {text_preview}")


=== ANGER (44 posts) ===
  [0.99] [caption+transcript] GYMSKIN doubled his AURA after they didnt BURN THE BEAN ???? #gymskin |  | These are... Too groovy. Look at these. They didn't fucking burn the bean. Look at these fucking...
  [0.98] [transcript_only] Please take a second and switch off the face ID on your phone if you're going through an airport. I don't care if you're white, I don't care if you're a citizen, I don't care if yo
  [0.98] [caption+transcript] dirty blonde core #dirtyblonde #blondehair #naturalhair |  | People die for this, people lie for this, people suck and fuck some guy for this, | pay the toll for this, sell their soul fo

=== ANTICIPATION (188 posts) ===
  [0.90] [caption_only] Floyd Mayweather Jr. and Manny Pacquiao will meet again in a professional boxing rematch on Sept. 19 at The Sphere in Las Vegas, sources told Andreas Hale. | It is not yet known what
  [0.89] [caption_only] Trae Young will make his team debut on Thursday against the Utah Jazz at home.

In [10]:
output_path = "../Outputs/emotion_cardiff_twitter_905.csv"
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: ../Outputs/emotion_cardiff_twitter_905.csv


# Emotion Model 3: Cardiff Twitter — Conclusion

**Model:** `cardiffnlp/twitter-roberta-base-emotion-multilabel-latest`
**Output labels:** 11 emotions — anger, anticipation, disgust, fear, joy, love, optimism, pessimism, sadness, surprise, trust
**Trained on:** 154 million tweets (social media domain match)
**Dataset:** 905 posts from 12 students, 797 had usable text

## What we did in plain terms

We ran every post through a third emotion model. This one is trained specifically on Twitter data, so it should understand social media language better than the others. It also has 11 emotions including useful categories like "anticipation" and "trust" that the other models didn't have. No "neutral" option, so every post gets an actual emotion label.

## What we found

Out of 797 usable posts:
- Joy: 362 (45%)
- Anticipation: 188 (24%)
- Optimism: 80 (10%)
- Sadness: 69 (9%)
- Anger: 44 (6%)
- Disgust: 29 (4%)
- Fear: 25 (3%)
- Love, Pessimism, Surprise, Trust: 0 each

The dominant emotion is heavily positive: joy + anticipation + optimism together account for 79% of posts. That fits with what we'd expect from a social media feed.

## Why this model is interesting

It introduces "anticipation" as a category and that catches a huge bucket (24% of posts). The other two models didn't have this label, so all the "I'm excited to announce I'll be joining X" and "I can't wait to see what this year brings" posts got forced into joy, neutral, or surprise. Now they have a home.

Sanity check examples look right for most categories:
- Anticipation: "I can't wait to see what this next year at Duke will bring"
- Optimism: "I pray something good happens in your life"
- Joy: "What an amazing night celebrating the 20th anniversary"
- Sadness: "Greys anatomy star Eric Dane passed away at age 53"

The model also produces much less false anger and fear than the first model (DistilRoBERTa). 44 anger posts here vs 146 fear and 46 anger before.

## The problems

Four out of 11 categories got zero posts (love, pessimism, surprise, trust). The model has these labels but never picks them as the dominant emotion. Joy absorbs what should be love. Anticipation absorbs what should be surprise. So the dominant-emotion view loses useful distinctions.

The keyword-reactivity issue is reduced but not gone. A comedic gym video got tagged as 99% anger because of profanity in the transcript. A TV show fan saying "let these people traumatize me for 10 more seasons" still got tagged as fear because of "traumatize."

## How the three emotion models compare

The same 797 posts get wildly different stories from each model.

| Bucket | DistilRoBERTa | GoEmotions | Cardiff Twitter |
|---|---|---|---|
| Positive (joy, love, gratitude, anticipation, etc.) | 23% | 23% | 79% |
| Negative (anger, fear, sadness, disgust) | 33% | 3% | 22% |
| Neutral | 35% | 63% | not available |

Each model has a different bias. DistilRoBERTa over-labels everything as negative. GoEmotions hides everything in neutral. Cardiff Twitter is more positive but still has anger false positives. There is no clean winner.

## Bottom line

Of the three transformer models we ran, Cardiff Twitter is the best fit for social media because it was trained on tweets and has "anticipation" as a useful category. It also avoids both the over-labeling problem of DistilRoBERTa and the over-neutralizing problem of GoEmotions.

But it still misfires on entertainment content with profanity or thematic emotion words. The keyword-reactivity issue is reduced, not solved.

The three-model comparison gives us strong evidence: no single transformer model handles social media emotion detection reliably. Different models give wildly different distributions of the same content. To get meaningful labels at the per-post level, we likely need either model agreement (use only when 2-3 models agree) or LLM-based classification that can read context.